In [2]:

# Definicion del tipo de problema (variable continua) y como se representa una solucion
from jmetal.core.problem import FloatProblem
from jmetal.core.solution import FloatSolution

# Utilizar un algoritmo genetico monoobjetivo
from jmetal.algorithm.singleobjective.genetic_algorithm import GeneticAlgorithm

# Operadores geneticos: crossover y mutacion
from jmetal.operator.crossover import SBXCrossover
from jmetal.operator.mutation import PolynomialMutation

# Como detener el algoritmo: iteraciones
from jmetal.util.termination_criterion import StoppingByEvaluations

# sklearn: Modelo: RandomForest; Evaluación: validación cruzada; Dataset: Breast Cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.datasets import load_breast_cancer

# Definicion del problema: Encontrar los mejores hiperparámetros para Random Forest
class RandomForestProblem(FloatProblem):

    def __init__(self):
        super().__init__()

        # Se definen los limites de cada variable:
        # [n_estimators, max_depth, min_samples_split, criterion]
        # criterion usa label encode: 0 = gini, 1 = entropy
        self.lower_bound = [10,  1,  2, 0]
        self.upper_bound = [200, 20, 20, 1]

        # Cargar en memoria el dataset
        self.data = load_breast_cancer()

    def number_of_variables(self):
        return 4

    def number_of_objectives(self):
        return 1

    def number_of_constraints(self):
        return 0

    def evaluate(self, solution):
        # Traducir cada variable continua al tipo que espera RandomForest
        n_estimators      = max(10, int(round(solution.variables[0])))
        max_depth         = max(1,  int(round(solution.variables[1])))
        min_samples_split = max(2,  int(round(solution.variables[2])))
        criterion         = "gini" if round(solution.variables[3]) == 0 else "entropy"

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            criterion=criterion,
            random_state=42     # Fijar semilla para que la evaluación sea reproducible
        )

        # Validación cruzada con 5 folds (más robusto que 3 porque el dataset es más complejo)
        score = cross_val_score(
            model,
            self.data.data,
            self.data.target,
            cv=5
        ).mean()

        # jMetal minimiza: se invierte el signo para maximizar accuracy
        solution.objectives[0] = -score
        return solution

    def create_solution(self):
        return FloatSolution(
            self.lower_bound,
            self.upper_bound,
            self.number_of_objectives()
        )

    def name(self):
        return "RandomForest: Optimizacion de hiperparametros"

In [ ]:

# 1) Crear el problema
problem = RandomForestProblem()

algorithm = GeneticAlgorithm(
    problem=problem,
    population_size=20,                                                      # 20 soluciones por generación (más que en ej. 2 por el espacio de búsqueda mayor)
    offspring_population_size=20,                                            # 20 nuevas soluciones por iteración
    mutation=PolynomialMutation(probability=0.25, distribution_index=20),    # 25% de probabilidad de mutación
    crossover=SBXCrossover(probability=0.9, distribution_index=20),          # 90% de probabilidad de cruce
    termination_criterion=StoppingByEvaluations(max_evaluations=200)         # Más evaluaciones porque el espacio de búsqueda es más grande
)

# El algoritmo genético realiza:
# 1) Genera soluciones aleatorias
# 2) Evalúa cada una (función evaluate)
# 3) Selecciona las mejores
# 4) Cruza y muta para generar nuevas soluciones
# 5) Repite hasta alcanzar max_evaluations
algorithm.run()

# Obtener la mejor solución encontrada
result = algorithm.result()

# Traducir las variables continuas a sus valores reales
best_n_estimators      = max(10, int(round(result.variables[0])))
best_max_depth         = max(1,  int(round(result.variables[1])))
best_min_samples_split = max(2,  int(round(result.variables[2])))
best_criterion         = "gini" if round(result.variables[3]) == 0 else "entropy"

print("Mejor n_estimators     :", best_n_estimators)
print("Mejor max_depth        :", best_max_depth)
print("Mejor min_samples_split:", best_min_samples_split)
print("Mejor criterion        :", best_criterion)
print("Accuracy               :", round(-result.objectives[0], 4))

[2026-05-04 00:05:14,174] [jmetal.core.algorithm] [DEBUG] Creating initial set of solutions...
[2026-05-04 00:05:14,175] [jmetal.core.algorithm] [DEBUG] Evaluating solutions...
[2026-05-04 00:05:15,952] [jmetal.core.algorithm] [DEBUG] Initializing progress...
[2026-05-04 00:05:15,953] [jmetal.core.algorithm] [DEBUG] Running main loop until termination criteria is met
[2026-05-04 00:05:31,673] [jmetal.core.algorithm] [DEBUG] Finished!


Mejor n_estimators     : 10
Mejor max_depth        : 5
Mejor min_samples_split: 2
Mejor criterion        : gini
Accuracy               : 0.9561
